# 04 - Training and evaluation

**Questions this notebook answers:** is the split honest, is there enough signal
to train, and is the model looking at the back?

Training is the cheapest step here. The expensive parts are the checks around it,
because a group-split classification problem with a few hundred crops fails in
quiet ways: a source shortcut, a degenerate split, a threshold tuned on test.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 60)
print("project root:", PROJECT_ROOT)

## Run configuration

In [ ]:
PREVIEW_ONLY = True     # flip to False to actually launch training
SAVE_OUTPUTS = False
MAX_SAMPLES = 6         # samples per error-analysis panel

MANIFEST_CSV = PROJECT_ROOT / "data" / "manifest.csv"
RUN_DIR = PROJECT_ROOT / "outputs" / "run01"
MODELS = ["geometry", "embedding", "fusion"]

In [ ]:
def guard_save(what: str) -> bool:
    """Refuse to write anything while the notebook is in preview mode."""
    if PREVIEW_ONLY or not SAVE_OUTPUTS:
        print(f"PREVIEW MODE - not writing {what}. Set PREVIEW_ONLY=False and SAVE_OUTPUTS=True to commit.")
        return False
    return True

## 1. Pre-flight: is this trainable?

Four things must hold before training is meaningful. Each failure below has a
real remedy, and none of the remedies is "re-roll the seed until it works" -
re-rolling the split after seeing labels is an indirect look at the test set.

In [ ]:
from cowarch.io import as_bool, read_manifest
from cowarch.splits import assert_no_group_leakage

manifest = read_manifest(MANIFEST_CSV)
accepted = manifest.loc[as_bool(manifest["accepted"])].copy()
labeled = accepted.loc[accepted["label"].isin(["arched", "normal"])]

problems = []

# (a) group leakage
try:
    assert_no_group_leakage(accepted, "video_id")
    print("PASS  no group leaks across splits")
except ValueError as error:
    problems.append(f"group leakage: {error}")

# (b) both classes in every split
table = pd.crosstab(labeled["split"], labeled["label"])
display(table)
for split in ["train", "val", "test"]:
    if split not in table.index or (table.loc[split] == 0).any():
        problems.append(
            f"split '{split}' does not hold both classes. "
            "Remedy: label more crops, or add source groups - not a new seed."
        )
    else:
        print(f"PASS  {split}: arched={table.loc[split].get('arched', 0)}, normal={table.loc[split].get('normal', 0)}")

# (c) enough groups behind the test split
test_groups = labeled.loc[labeled["split"] == "test", "video_id"].nunique()
if test_groups < 3:
    problems.append(f"test rests on {test_groups} groups; 3-4 minimum for any generalisation claim")
else:
    print(f"PASS  test spans {test_groups} groups")

# (d) keypoint coverage for Model A
has_kp = labeled["keypoints_json"].astype(str).str.strip().ne("")
print(f"INFO  keypoints on {int(has_kp.sum())}/{len(labeled)} labeled crops")

print()
if problems:
    for problem in problems:
        print("BLOCKER:", problem)
else:
    print("pre-flight clear")

### Class balance is not the only thing to check

If one source dominates a class, the model can score well by recognising the
source. The table below is the one to worry about: a class that lives almost
entirely in one video is a shortcut waiting to be learned.

In [ ]:
if len(labeled):
    by_source = pd.crosstab(labeled["source_id"], labeled["label"])
    by_source["arched_share"] = (by_source.get("arched", 0) / by_source.sum(axis=1)).round(2)
    display(by_source.sort_values("arched_share", ascending=False))

    fig, ax = plt.subplots(figsize=(8, 3.2))
    by_source[["arched", "normal"]].plot.barh(stacked=True, ax=ax,
                                              color=["#d1495b", "#2e86ab"])
    ax.set_xlabel("labeled crops"); ax.set_title("label composition per source")
    plt.tight_layout(); plt.show()

    dominated = by_source[(by_source["arched_share"] > 0.9) | (by_source["arched_share"] < 0.1)]
    if len(dominated):
        print(f"{len(dominated)} source(s) carry effectively one class. "
              "Note this in limitations: source identity partly predicts the label.")

## 2. Train

The backend fits three models on the same splits:

| Model | Features | Reads as |
|---|---|---|
| A `geometry` | 5 interpretable dorsal measurements | can a ruler do this? |
| B `embedding` | frozen ResNet18, 512-D | can generic visual features do this? |
| C `fusion` | A + B | do they carry different information? |

Hyperparameter `C` is picked on validation PR-AUC and the decision threshold on
validation F1. Test is touched once, at the end.

In [ ]:
import subprocess

command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "05_train.py"),
    "--manifest", str(MANIFEST_CSV),
    "--output-dir", str(RUN_DIR),
    "--models", *MODELS,
]
print(" ".join(command))
if guard_save(f"models and metrics under {RUN_DIR}"):
    result = subprocess.run(command, cwd=PROJECT_ROOT)
    print("exit code:", result.returncode)

## 3. Results

Accuracy is not reported. With an imbalanced, non-representative sample it is the
metric most likely to flatter a model that has learned the majority class.

In [ ]:
import json

metrics_path = RUN_DIR / "metrics.json"
if not metrics_path.exists():
    print(f"no metrics at {metrics_path} - run section 2 with SAVE_OUTPUTS=True")
    metrics = {}
else:
    metrics = json.loads(metrics_path.read_text())
    rows = []
    for name, payload in metrics.items():
        for split in ["validation", "test"]:
            rows.append({
                "model": name, "split": split,
                **{k: round(v, 3) for k, v in payload[split].items() if isinstance(v, float)},
                "n": payload[split]["n"],
            })
    display(pd.DataFrame(rows).set_index(["model", "split"]))

In [ ]:
predictions_path = RUN_DIR / "predictions.csv"
predictions = pd.read_csv(predictions_path) if predictions_path.exists() else pd.DataFrame()

if len(predictions):
    from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay, confusion_matrix

    test_predictions = predictions[predictions["split"] == "test"]
    models_present = test_predictions["model"].unique()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name in models_present:
        subset = test_predictions[test_predictions["model"] == name]
        PrecisionRecallDisplay.from_predictions(subset["target"], subset["probability_arched"],
                                                name=name, ax=axes[0])
        RocCurveDisplay.from_predictions(subset["target"], subset["probability_arched"],
                                         name=name, ax=axes[1])
    axes[0].set_title("Precision-Recall (test)"); axes[1].set_title("ROC (test)")
    axes[1].plot([0, 1], [0, 1], "k--", linewidth=0.8)
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, len(models_present), figsize=(4 * len(models_present), 3.4))
    for ax, name in zip(np.atleast_1d(axes), models_present):
        subset = test_predictions[test_predictions["model"] == name]
        matrix = confusion_matrix(subset["target"], (subset["prediction"] == "arched").astype(int), labels=[0, 1])
        ax.imshow(matrix, cmap="Blues")
        for (i, j), value in np.ndenumerate(matrix):
            ax.text(j, i, str(value), ha="center", va="center",
                    color="white" if value > matrix.max() / 2 else "black", fontsize=13)
        ax.set_xticks([0, 1], ["pred normal", "pred arched"])
        ax.set_yticks([0, 1], ["true normal", "true arched"])
        ax.set_title(name, fontsize=10)
    plt.tight_layout(); plt.show()

### Confidence intervals

With 30-40 test crops a point estimate is nearly meaningless on its own. The
bootstrap interval below is usually wide enough to make that obvious, which is
the honest way to present a PoC.

In [ ]:
if len(predictions):
    from sklearn.metrics import average_precision_score

    def bootstrap_pr_auc(y, p, n_boot=2000, seed=0):
        rng = np.random.default_rng(seed)
        y = np.asarray(y); p = np.asarray(p)
        scores = []
        for _ in range(n_boot):
            idx = rng.integers(0, len(y), len(y))
            if len(np.unique(y[idx])) < 2:
                continue
            scores.append(average_precision_score(y[idx], p[idx]))
        return np.percentile(scores, [2.5, 97.5]) if scores else (np.nan, np.nan)

    for name in test_predictions["model"].unique():
        subset = test_predictions[test_predictions["model"] == name]
        point = average_precision_score(subset["target"], subset["probability_arched"])
        low, high = bootstrap_pr_auc(subset["target"], subset["probability_arched"])
        print(f"{name:10s} PR-AUC {point:.3f}  95% CI [{low:.3f}, {high:.3f}]  (n={len(subset)})")

## 4. Error analysis

Not "how many did it get wrong" but "what kind of thing does it get wrong". The
geometry overlay is drawn on each error so you can see whether the model was
unreasonable or the label was.

In [ ]:
import cv2
from cowarch.geometry import decode_keypoints
from cowarch.io import resolve_data_path
from cowarch.viz import panel_keypoint_geometry

def show_errors(model_name: str, kind: str = "false_positive", limit: int = MAX_SAMPLES):
    subset = test_predictions[test_predictions["model"] == model_name].copy()
    if kind == "false_positive":
        errors = subset[(subset["target"] == 0) & (subset["prediction"] == "arched")]
        errors = errors.sort_values("probability_arched", ascending=False)
    else:
        errors = subset[(subset["target"] == 1) & (subset["prediction"] == "normal")]
        errors = errors.sort_values("probability_arched")
    if not len(errors):
        print(f"{model_name}: no {kind}s")
        return
    errors = errors.head(limit)
    keys = manifest.set_index("sample_id")
    columns = min(3, len(errors))
    rows = int(np.ceil(len(errors) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4.2 * columns, 3.2 * rows), squeeze=False)
    for ax, (_, row) in zip(axes.ravel(), errors.iterrows()):
        crop = cv2.imread(str(resolve_data_path(str(row["crop_path"]), MANIFEST_CSV)))
        points = decode_keypoints(keys.loc[row["sample_id"], "keypoints_json"])
        panel_keypoint_geometry(ax, crop, points)
        ax.set_title(f"{row['sample_id']}\ntrue={row['label']} p={row['probability_arched']:.2f}", fontsize=8)
    for ax in axes.ravel()[len(errors):]:
        ax.axis("off")
    fig.suptitle(f"{model_name}: worst {kind}s", y=1.02)
    plt.tight_layout(); plt.show()

if len(predictions):
    for name in test_predictions["model"].unique():
        show_errors(name, "false_positive")
        show_errors(name, "false_negative")

### Errors by slice

The overall number hides where the model fails. If almost all errors sit in
`review_oblique` frames, the finding is about view filtering, not posture.

In [ ]:
if len(predictions):
    joined = test_predictions.merge(
        manifest[["sample_id", "view_hint", "source_id", "aspect_ratio", "blur_laplacian_var"]],
        on="sample_id", how="left", suffixes=("", "_m"),
    )
    joined["correct"] = (joined["prediction"] == joined["label"])
    for column in ["view_hint", "source_id"]:
        if column in joined.columns:
            slice_table = joined.groupby(["model", column])["correct"].agg(["mean", "size"])
            slice_table.columns = ["accuracy_in_slice", "n"]
            display(slice_table.round(3))

## 5. Is the model looking at the back?

The decisive check. Grey out one patch at a time, re-embed, and see how far the
prediction moves. If the sensitivity sits on the barn rail rather than the dorsal
line, the metrics above describe a source detector, not a posture detector.

`topline_attention_ratio` reduces this to one number: the share of positive
sensitivity falling in a band just under the silhouette's topline.

In [ ]:
import joblib
from cowarch.explain import embedding_score_fn, occlusion_map, topline_attention_ratio, upsample_map

bundle_path = RUN_DIR / "embedding.joblib"
if not bundle_path.exists():
    print("no embedding model - train it first (section 2)")
else:
    bundle = joblib.load(bundle_path)
    score_fn = embedding_score_fn(bundle["pipeline"])

    candidates = test_predictions[test_predictions["model"] == "embedding"].head(3)
    for _, row in candidates.iterrows():
        crop = cv2.imread(str(resolve_data_path(str(row["crop_path"]), MANIFEST_CSV)))
        mask_path = manifest.set_index("sample_id").loc[row["sample_id"], "mask_path"]
        mask = None
        if str(mask_path).strip():
            raw = cv2.imread(str(resolve_data_path(str(mask_path), MANIFEST_CSV)), cv2.IMREAD_GRAYSCALE)
            mask = None if raw is None else raw > 127

        sensitivity = occlusion_map(crop, score_fn, grid=(6, 8))
        heat = upsample_map(sensitivity, crop.shape[:2])

        fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
        axes[0].imshow(crop[:, :, ::-1]); axes[0].set_title("crop"); axes[0].axis("off")
        axes[1].imshow(heat, cmap="RdBu_r", vmin=-np.abs(heat).max(), vmax=np.abs(heat).max())
        axes[1].set_title("occlusion sensitivity"); axes[1].axis("off")
        axes[2].imshow(crop[:, :, ::-1]); axes[2].imshow(heat, cmap="RdBu_r", alpha=0.5,
                       vmin=-np.abs(heat).max(), vmax=np.abs(heat).max())
        ratio = topline_attention_ratio(heat, mask) if mask is not None else float("nan")
        axes[2].set_title(f"overlay - topline attention {ratio:.2f}"); axes[2].axis("off")
        fig.suptitle(f"{row['sample_id']}  true={row['label']}  p={row['probability_arched']:.2f}", y=1.03)
        plt.tight_layout(); plt.show()

## 6. Produce the report

`scripts/06_report.py` writes `REPORT.md` with metrics, figures and the mandatory
interpretation limits.

In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "06_report.py"),
    "--run-dir", str(RUN_DIR),
    "--manifest", str(MANIFEST_CSV),
]
print(" ".join(command))
if guard_save(f"{RUN_DIR}/REPORT.md"):
    result = subprocess.run(command, cwd=PROJECT_ROOT)
    print("exit code:", result.returncode)

## What may be claimed

Supported by this run:

> A model separates human-annotated lateral arched-back posture labels with
> test PR-AUC X (95% CI [a, b]) over N crops from G independent source groups.

Not supported, whatever the numbers say:

- lameness or disease diagnosis
- a locomotion score
- field generalisation - one internal split does not cross farms, cameras or breeds
- comparison against Hoffman's clinical 0.63/0.64, which measured a different thing

If topline attention was low, say so. A model with good metrics and evidence
sitting on the background is a negative result, and reporting it as such is worth
more than the number.